# Head konteks tetap: validation dengan input palette6

Buka dari root **SongMap**, lalu **Run All**. Python dipanggil dari sibling **Section-Annotations/.venv-songformer311** secara eksplisit.

Kita memakai **snapshot head konteks epoch 5 yang sama**, dengan fitur encoder MusicFM/MuQ palette6 dan probabilitas head Core ML palette6 dari laporan validation tersimpan. Tidak ada training, perubahan bobot, atau pemilihan parameter baru.

Perbandingan memakai **32 lagu validation yang sama**: fitur asli + konteks, palette6 tanpa konteks, dan palette6 + konteks. Bandingkan kandidat dengan hasil validation asli (**macro F1 0,7755**), bukan angka 80% pada 34 lagu test.

Head konteks pada eksperimen ini **masih PyTorch**. Encoder dan head Core ML lama tidak dijalankan ulang karena fitur serta probabilitasnya sudah tersimpan. Ini pemeriksaan kompatibilitas input palette6 sebelum konversi head konteks ke Core ML; belum merupakan pengujian end-to-end iPhone.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json

SONGMAP_ROOT=Path.cwd().resolve()
assert (SONGMAP_ROOT/'scripts/evaluate_palette6_context.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=(SONGMAP_ROOT/'../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'

# Snapshot dipilih dari folder evaluasi terakhir, bukan berdasarkan skor test.
files=[p for p in (SONGMAP_ROOT/'outputs.local').glob('frozen-context-test-*/selection.json')
       if (p.parent/'frozen-context.pt').is_file()]
assert files, 'Snapshot dari section_context_test.ipynb belum ditemukan'
FROZEN=max(files,key=lambda p:p.stat().st_mtime).parent.resolve()
selection=json.loads((FROZEN/'selection.json').read_text())
assert selection['context_epoch']==5, 'Arahkan FROZEN ke snapshot epoch 5 yang sudah dipilih'

proofs=list((SONGSTRUCT_ROOT/'outputs.local').glob('muq-compression-*/palette6-all-validation.json'))
assert proofs, 'Laporan palette6-all-validation.json belum ditemukan'
PALETTE6_REPORT=max(proofs,key=lambda p:p.stat().st_mtime).resolve()
original_cache=SONGMAP_ROOT/'outputs.local/section-context-cache-v1'
palette_cache=SONGMAP_ROOT/'outputs.local/section-context-palette6-cache-v1'
output=SONGMAP_ROOT/'outputs.local'/('context-palette6-validation-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([str(RUN_PYTHON),'-c',
    "import sys,numpy,torch; print('Python:',sys.executable); print('NumPy:',numpy.__version__,'Torch:',torch.__version__)"],check=True)
print('Model terkunci:',FROZEN)
print('Laporan palette6:',PALETTE6_REPORT)
print('Cache ringkasan palette6:',palette_cache)
print('Output:',output)


Script memeriksa hash snapshot dan rantai laporan MusicFM/MuQ/head palette6, menghasilkan ulang skor head konteks pada fitur asli, serta memeriksa ulang skor baseline dari probabilitas palette6 tersimpan. Setelah itu, cache encoder palette6 dibaca sekali untuk membuat ringkasan audio semua section.

Jika terputus, jalankan ulang **cell berikut dengan output yang sama**. Cache ringkasan yang selesai akan dipakai kembali. Encoder, audio decoder, dan head Core ML lama tidak dipanggil; tidak perlu mengulang training.


In [ ]:
def run_validation():
    if (output/'validation-all.json').is_file():
        print('Evaluasi sudah selesai:',output/'validation-all.json')
        return
    command=[str(RUN_PYTHON),'-u',str(SONGMAP_ROOT/'scripts/evaluate_palette6_context.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--frozen-evaluation',str(FROZEN),
        '--palette6-report',str(PALETTE6_REPORT),'--original-cache-dir',str(original_cache),
        '--cache-dir',str(palette_cache),'--output-dir',str(output)]
    log=Path(str(output)+'.log');log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=SONGMAP_ROOT,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log}. Cache dan snapshot tetap tersedia.')

run_validation()


In [ ]:
report=json.loads((output/'validation-all.json').read_text())
assert report['full_validation']
print('Validation:',report['tracks'],'lagu | epoch tetap:',report['selected_epoch'])
for key in ('original_features_with_context','palette6_baseline','palette6_features_with_context'):
    score=report[key]
    print(f"{key}: accuracy={score['section_accuracy']:.2%}; macro F1={score['macro_f1_present']:.4f}")
print('F1 per kelas (asli+konteks | palette6 | palette6+konteks):')
keys=('original_features_with_context','palette6_baseline','palette6_features_with_context')
for label in report[keys[0]]['label_order']:
    scores=[report[key]['per_class'][label]['f1'] for key in keys]
    print(f"  {label}: {scores[0]:.4f} | {scores[1]:.4f} | {scores[2]:.4f}")
print('Delta F1 vs fitur asli + konteks:',report['macro_f1_delta_from_original_context'])
print('Delta F1 vs baseline palette6:',report['macro_f1_delta_from_palette6_baseline'])
print('Kesamaan label dengan konteks pada fitur asli:',f"{report['section_agreement_with_original_context']:.2%}")
print('Kirim file:',output/'validation-all.json')
print('Head konteks masih PyTorch; belum dikonversi atau dipasang ke aplikasi.')


Bagikan **validation-all.json**. Laporan memuat perubahan section terhadap baseline palette6 dan terhadap head konteks pada fitur asli, sehingga efek input terkompresi bisa dipisahkan dari manfaat head konteks.

Boundary dan label manual tetap dipertahankan. Hasil tahap ini tidak menetapkan peningkatan kecepatan, peak memory iPhone, atau akurasi boundary.
